# Study 910 — Managed-Distribution CEF 🎁

**Do persistent-discount closed-end funds with a big "managed distribution" hand you the
discount pull *and* the payout — or just a levered-beta clone with a yield sticker?**

The pitch: a closed-end fund (CEF) trades below the value of what it holds (buy a dollar of assets
for ~90 cents — the *discount pull*) *and* pays a fat, level distribution (8–14 %/yr). Sounds like
free double-carry. The sceptic's counter — the mREIT lesson of
[611](../../611-mreit-carry/) — is that the fat payout is often your own capital handed back
(*return of capital*), and the leverage inside the wrapper is financed at short rates, so **NAV
erosion + leverage cost can eat the whole thing**.

We test the buyer's bottom line on **PCEF** (the CEF-of-CEFs) and an equal-weight basket of four
large CEFs (**PDI, UTF, BST, RQI**) vs **SPY**, everything **excess of cash (BIL)**, on
total-return tape (2014-11 → 2026-06 for the basket; PCEF from 2010).

*Numbers below are the frozen headline (`docs/results.md`, fingerprint `cae85d4d2cca`, as-of
2026-06-30); the live cells run the fast synthetic control. Survivorship + short basket history
bias the magnitude upward.*


## 1. The two things a CEF supposedly gives you

**The discount pull.** If a fund holding \$100 of assets trades at \$90, you own \$100 of stuff for \$90 — and if the discount ever closes, you pocket the gap.

**The payout.** A *managed distribution plan* pays a fixed, fat yield on a schedule. The catch the brochure buries: when the fund hasn't *earned* that much, the extra is **return of capital** — literally your own money handed back, while the NAV quietly shrinks. A 12 % 'yield' that is half return-of-capital is a 6 % yield plus a slow leak. Because we use **total-return** prices (distributions reinvested), our numbers see through the label to the real economic return.

In [1]:
R = dict(b_exret=88.3, b_t=2.63, b_sharpe=0.61, spy_sharpe=0.82, b_adv=-0.21,
         b_alpha=-1.6, b_talpha=-0.59, b_beta=1.0, b_maxdd=-28, rqi_maxdd=-87)
print('CEF basket, excess of cash:  %+.1f bps/mo  (HAC t = %+.2f)'
      % (R['b_exret'], R['b_t']))
print('  -> a REAL payout: the return clears cash, t > 2')
print('excess-of-cash Sharpe:  basket %.2f   vs   SPY %.2f   (advantage %+.2f)'
      % (R['b_sharpe'], R['spy_sharpe'], R['b_adv']))
print('  -> but risk-adjusted it TRAILS the index it is sold against')
print('CAPM alpha vs SPY:  %+.1f%%/yr (t = %+.2f),  beta = %.2f  (levered!)'
      % (R['b_alpha'], R['b_talpha'], R['b_beta']))
print('worst real-estate CEF (RQI) max drawdown: %d%%' % R['rqi_maxdd'])

CEF basket, excess of cash:  +88.3 bps/mo  (HAC t = +2.63)
  -> a REAL payout: the return clears cash, t > 2
excess-of-cash Sharpe:  basket 0.61   vs   SPY 0.82   (advantage -0.21)
  -> but risk-adjusted it TRAILS the index it is sold against
CAPM alpha vs SPY:  -1.6%/yr (t = -0.59),  beta = 1.00  (levered!)
worst real-estate CEF (RQI) max drawdown: -87%


## 2. Is the machinery honest? A live synthetic control

We build a toy 'CEF' = a **levered** claim on a market factor (β = 1.1) **plus** a structural carry we can dial, **minus** a return-of-capital leak we can dial. Three worlds, all offline:

* **null** — pure levered beta, no carry: the alpha test must stay silent;
* **planted** — a genuine +5 %/yr net carry: the alpha must light up;
* **return-of-capital trap** — a fat carry that is *entirely* leaked back (carry = leak): net zero, so — like the worst real CEFs — it must produce **no alpha**.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from md_cef import data, strategy as st
for carry, leak, tag in [(0.0,0.0,'null (levered beta)'),
                         (0.05,0.0,'planted +5%/yr carry'),
                         (0.05,0.05,'return-of-capital trap')]:
    d = st.synthetic_detect(data.synthetic_world(carry_annual=carry, roc_leak_annual=leak, seed=910))
    fires = 'FIRES' if abs(d['t_alpha'])>=2 else 'silent'
    print(f"{tag:<24s}: alpha {d['alpha_ann_pct']:+.2f}%/yr (t {d['t_alpha']:+.2f})  beta {d['beta']:.2f}  -> {fires}")

null (levered beta)     : alpha -1.46%/yr (t -0.86)  beta 1.11  -> silent
planted +5%/yr carry    : alpha +3.54%/yr (t +2.10)  beta 1.11  -> FIRES
return-of-capital trap  : alpha -1.46%/yr (t -0.86)  beta 1.11  -> silent


## 3. The honest verdict — half-true

The payout is **real**: the basket earns **+88.3 bps/mo excess of cash** (HAC *t* = +2.63), and unlike a blown-up mREIT its total return genuinely clears cash. So the sticker isn't pure fiction.

But the **edge over the asset class isn't there**. Risk-adjusted, the basket's excess-of-cash Sharpe (**0.61**) **trails SPY's (0.82)**, the CAPM alpha is -1.6 %/yr (*t* = -0.59), and β ≈ 1.00 tells the story: you're holding **levered equity beta** dressed as income. When rates jumped in 2022 the whole thing fell **-24.4 %** in a year. **Signal: Weak** (real payout, no asset-class edge), **Tradability: Mirage** (a hidden levered beta — not costs — erases it; you'd do better just holding SPY).